Question 1: Data Cleaning and Basic Analysis

Using the provided dataset 'sales_data.csv', perform the following tasks:

1. Load and examine the dataset
2. Clean the data:
   - Handle missing values in the 'price' column (replace with mean)
   - Handle missing values in the 'category' column (replace with mode)
   - Convert 'sale_date' to datetime
3. Create a summary of:
   - Total number of sales per category
   - Average price per category
   - Number of missing values handled

The cleaned dataset should be ready for further analysis.

In [1]:
import pandas as pd
import numpy as np

from datetime import date

In [2]:
# a few considerations:
# 1. I moved the data to a folder called "raw", as it is a good practice
# to maintain raw data and treated data in separate locations. I created 
# a folder called "curated", which later the treated data will be saved to

FILE_NAME = 'sales_data.csv'

# first, let's load the raw data into a dataframe
sales_df = pd.read_csv(f'../data/raw/{FILE_NAME}')

# before we deal with missing values, let's count how many empty cells we have 
# for price and category. Since Nan == Nan always yields False, we just need 
# to check for rows for which the category or the price is different from itself
missing_category = len(sales_df[sales_df['category'] != sales_df['category']])
missing_price = len(sales_df[sales_df['price'] != sales_df['price']])

print(f"Rows with missing category: {missing_category}")
print(f"Rows with missing price: {missing_price}")

# let us now get the category mode. I do this first because for the 
# missing prices I want to fill them with the mean for that particular category

# .mode() always return a Series, even when there's just one value
# there are cases where there are more than one mode, but in this case, it is 
# just one, so we can just select it

category_mode = sales_df['category'].mode()[0]
# fill missing values with mode
sales_df['category'] = sales_df['category'].fillna(category_mode)



Rows with missing category: 30
Rows with missing price: 49



in a real-life dataset, it is expected that purchases from different 
categories have values for the total that differ, statistically speaking.
In our case, it is expected that, since books are cheaper than electronics,
the mean total for book purchases is lower than the mean for electronics purchases. 
Hence, we should fill missing values with the mean for that category, 
and not with the overall mean. As it can be seen in the cell above, this is not the case, but I wanted 
to treat this dataset as if it was a real one.

In [3]:
sales_df.groupby("category").describe()[['price'][0]][['mean', 'std']]

,mean,std
category,,
Books,101.690430,31.559471
Clothing,103.822460,30.128680
Electronics,100.468345,27.130517
Home & Garden,102.661415,33.920833
Sports,101.188330,29.640805


Now, let's fill the price by the category mean and convert `sale_date` to datetime.

In [4]:
sales_df['price'] = sales_df['price'].fillna(sales_df.groupby('category')['price'].transform('mean'))
sales_df['sale_date'] = sales_df['sale_date'].apply(lambda x: date(*[int(d) for d in x.split('-')]))

In [5]:
# This is the final dataset. With filled values for category and price

sales_df.sample(30)

,sale_id,sale_date,category,price,quantity
998,999,2025-09-25,Home & Garden,94.585603,9
690,691,2024-11-21,Sports,100.845435,1
75,76,2023-03-17,Clothing,103.822460,8
368,369,2024-01-04,Home & Garden,102.661415,1
952,953,2025-08-10,Home & Garden,53.123624,5
280,281,2023-10-08,Clothing,94.986458,8
97,98,2023-04-08,Clothing,111.974083,7
544,545,2024-06-28,Electronics,25.017829,1
253,254,2023-09-11,Books,160.306136,5
181,182,2023-07-01,Books,120.445021,5


In [6]:
# average price and number of sales by category 
sales_df.groupby("category").describe()[['price'][0]][['mean', 'count']]

,mean,count
category,,
Books,101.690430,179.0
Clothing,103.822460,185.0
Electronics,100.468345,238.0
Home & Garden,102.661415,201.0
Sports,101.188330,197.0


In [7]:
# Finally, we can save the dataset to the curated layer
sales_df.to_csv(f"../data/curated/{FILE_NAME}", index=False)